## 使用 Transformers 执行本地大语言模型推理

本文件演示通过 Hugging Face `transformers` 加载 `Qwen/Qwen2.5-0.5B-Instruct` 并完成一次对话生成的完整流程。

### 核心概念

| 术语 | 说明 |
| --- | --- |
| Tokenizer | 文本与 token ID 之间的双向映射；模型仅接受数值输入 |
| Causal LM | 因果语言模型，依据已有序列自回归预测下一个 token |
| Instruct 模型 | 经指令微调，具备指令遵循与对话能力；基座模型仅适用于续写 |
| Chat Template | 模型训练时约定的对话序列化格式，须使用模型自带模板 |

### 依赖

```bash
pip install torch transformers accelerate
```

| 包 | 作用 |
| --- | --- |
| `torch` | 张量计算与硬件加速 |
| `transformers` | 模型与分词器的统一加载接口 |
| `accelerate` | 支持 `device_map="auto"` 的自动设备分配 |

模型权重约 1 GB，首次运行时自动下载并缓存。下载缓慢时可设置环境变量 `HF_ENDPOINT` 指向镜像站点。

### 导入依赖

| 名称 | 说明 |
| --- | --- |
| `AutoTokenizer` | 依据模型标识自动匹配对应的分词器实现 |
| `AutoModelForCausalLM` | 因果语言模型类，用于自回归生成 |

不同任务需选用对应的模型类：序列分类使用 `AutoModelForSequenceClassification`，掩码语言建模使用 `AutoModelForMaskedLM`，二者均不适用于文本生成。

In [19]:
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer

### 指定模型标识

`Qwen/Qwen2.5-0.5B-Instruct` 命名解析：

| 片段 | 含义 |
| --- | --- |
| `Qwen2.5` | 模型系列与版本 |
| `0.5B` | 参数量约 5 亿；规模越小，资源占用越低 |
| `Instruct` | 经指令微调，适用于问答场景 |

分词器与模型须使用同一标识，二者不匹配会导致编码结果错误。

In [20]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

### 加载分词器

分词器负责文本与 token ID 的双向转换：

- 编码：文本 → ID 序列
- 解码：ID 序列 → 文本

token 不等同于字符或单词；英文常见单词通常为单个 token，中文单字一般对应 1~2 个 token。

In [21]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

### 加载模型

| 参数 | 说明 |
| --- | --- |
| `torch_dtype="auto"` | 采用模型推荐的权重精度，通常为 `bfloat16` 或 `float16`，显存占用约为 `float32` 的一半 |
| `device_map="auto"` | 由 `accelerate` 自动将各层分配至可用设备（CUDA / MPS / CPU） |

资源受限时的可选方案：换用更小规模的模型、启用 4-bit 量化（`load_in_4bit=True`，依赖 `bitsandbytes`），或显式指定 `device_map="cpu"`。

当前设备可通过 `model.device` 查看。

In [22]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto",
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

### 构造对话消息

对话以带角色的消息序列表示：

| `role` | 用途 |
| --- | --- |
| `system` | 设定助手行为准则，置于序列首位 |
| `user` | 用户输入 |
| `assistant` | 模型历史回复，多轮对话需保留 |

多轮对话示例：

```python
messages = [
    {"role": "system", "content": "You are a concise, high-efficiency technical assistant."},
    {"role": "user", "content": "什么是向量数据库？"},
    {"role": "assistant", "content": "用于存储与检索向量表示的数据库……"},
    {"role": "user", "content": "它与关系型数据库有何区别？"},
]
```

In [23]:
prompt = (
    "In one sentence, explain what a large language model is to a backend developer."
)
messages = [
    {
        "role": "system",
        "content": "You are a concise, high-efficiency technical assistant.",
    },
    {"role": "user", "content": prompt},
]

### 应用聊天模板并分词

#### apply_chat_template

将消息列表序列化为模型训练时使用的单段文本。Qwen 系列格式如下：

```
<|im_start|>system
You are a concise, high-efficiency technical assistant.<|im_end|>
<|im_start|>user
In one sentence, explain what a large language model is to a backend developer.<|im_end|>
<|im_start|>assistant
```

| 参数 | 说明 |
| --- | --- |
| `tokenize=False` | 返回字符串而非 ID 序列，便于检查 |
| `add_generation_prompt=True` | 末尾追加 `<|im_start|>assistant\n`，标识生成起始位置；缺失将显著劣化输出质量 |

#### 分词

| 要点 | 说明 |
| --- | --- |
| 列表输入 | 传入 `[text]` 以保留 batch 维度，支持批量推理 |
| `return_tensors="pt"` | 返回 PyTorch 张量 |
| 返回值 | 含 `input_ids` 与 `attention_mask` 的 `BatchEncoding` |
| `.to(model.device)` | 输入须与模型位于同一设备，否则触发设备不一致错误 |

In [24]:
text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

### 生成与解码

#### 生成参数

| 参数 | 说明 |
| --- | --- |
| `max_new_tokens=256` | 新生成 token 的数量上限，不包含输入部分；达到上限将被截断 |
| `do_sample=True` | 启用随机采样；默认 `False` 为贪心解码，输出确定但易重复 |
| `temperature=0.7` | 调整概率分布的平滑程度，值越低输出越确定；仅在采样模式下生效 |

#### 截取新生成内容

`generate` 返回输入与输出的拼接序列，需按输入长度切片，仅保留新生成部分。

#### 解码

`batch_decode` 将 ID 序列还原为文本，`skip_special_tokens=True` 用于移除 `<|im_end|>` 等控制符。

#### 其他常用参数

| 参数 | 说明 |
| --- | --- |
| `top_p` / `top_k` | 核采样与 top-k 采样，缩小候选集合 |
| `repetition_penalty` | 抑制重复内容 |
| `num_beams` | 束搜索，与采样模式互斥 |

生成原理：每步预测下一个 token 并将其追加至序列，循环执行直至产出结束符或达到长度上限，即自回归生成。

In [25]:
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.7,
)

generated_ids = [
    output_ids[len(input_ids) :]
    for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

### 输出

Notebook 中末行变量以 `repr` 形式呈现。如需渲染其中的 Markdown 结构，使用 `print(response)` 或 `IPython.display.Markdown`。

In [26]:
response

'A large language model is a powerful computational entity that can generate human-like text, enabling the development of conversational AI systems and applications.'

### 参数实验

| 调整项 | 预期效果 |
| --- | --- |
| `do_sample=False` | 多次运行结果一致 |
| `temperature` 取 0.1 与 1.2 | 对比确定性与多样性 |
| 修改 `system` 提示词 | 改变回答风格 |
| 追加 `assistant` 与 `user` 消息 | 实现多轮上下文 |

### 常见问题

| 现象 | 处理方式 |
| --- | --- |
| 下载缓慢或超时 | 设置 `HF_ENDPOINT` 指向镜像站点 |
| 显存不足 | 换用更小模型、启用 4-bit 量化或指定 `device_map="cpu"` |
| 答非所问 | 确认使用 Instruct 模型并设置 `add_generation_prompt=True` |
| 输出被截断 | 调大 `max_new_tokens` |
| 结果不可复现 | `do_sample=True` 的正常表现；可关闭采样或固定随机种子 |

### 延伸方向

| 方向 | 相关组件 |
| --- | --- |
| 高阶封装 | `transformers.pipeline` |
| 流式输出 | `TextStreamer` |
| 参数高效微调 | `peft`（LoRA）、`Trainer` |
| 检索增强生成 | 向量数据库、`sentence-transformers` |